## KNN解决FBLocation问题

**数据集下载：**[Facebook V：预测签到时间 |卡格尔 --- Facebook V: Predicting Check Ins | Kaggle](https://www.kaggle.com/competitions/facebook-v-predicting-check-ins/data)

In [2]:
import pandas as pd

# 假设训练集为"train.csv"，位于"FBLocation"文件夹内
train_df = pd.read_csv('FBLocation/train.csv')

# 使用 query 方法筛选 x 在 (1.0, 1.25) 且 y 在 (2.5, 2.75) 之间的数据，并用 copy 防止链式赋值警告
filtered_df = train_df.query('1.0 < x < 1.25 and 2.5 < y < 2.75').copy()

# 将 time 列视为分钟，并转换为时间格式（假设时间原点为1970-01-01 00:00）
filtered_df['time'] = pd.to_datetime(filtered_df['time'], unit='m')

# 查看筛选后的数据集及其形状
print(filtered_df.head())
print("filtered_df shape:", filtered_df.shape)




      row_id       x       y  accuracy                time    place_id
600      600  1.2214  2.7023        17 1970-02-15 09:40:00  6683426742
957      957  1.1832  2.6891        58 1971-06-30 11:10:00  6683426742
4345    4345  1.1935  2.6550        11 1970-10-05 20:02:00  6889790653
4735    4735  1.1452  2.6074        49 1970-12-24 15:03:00  6822359752
5580    5580  1.0089  2.7287        19 1971-05-24 14:50:00  1527921905
filtered_df shape: (17710, 6)


In [3]:
# 从 time 提取星期、小时和天
filtered_df['weekday'] = filtered_df['time'].dt.weekday
filtered_df['hour'] = filtered_df['time'].dt.hour
filtered_df['day'] = filtered_df['time'].dt.day

# 保留出现次数大于3的place_id，其余均删除
place_counts = filtered_df['place_id'].value_counts()
filtered_df = filtered_df[filtered_df['place_id'].isin(place_counts[place_counts > 3].index)].copy()
print("filtered_df shape after removing rare place_id:", filtered_df.shape)

# 移除 row_id，time，place_id 作为特征，place_id 作为标签
X = filtered_df.drop(['row_id', 'time', 'place_id'], axis=1)
y = filtered_df['place_id']
print("特征值 (X) 预览：")
print(X.head())
print("特征形状：", X.shape)

print("标签值 (y) 预览：")
print(y.head())
print("标签形状：", y.shape)



filtered_df shape after removing rare place_id: (16918, 9)
特征值 (X) 预览：
           x       y  accuracy  weekday  hour  day
600   1.2214  2.7023        17        6     9   15
957   1.1832  2.6891        58        2    11   30
4345  1.1935  2.6550        11        0    20    5
4735  1.1452  2.6074        49        3    15   24
5580  1.0089  2.7287        19        0    14   24
特征形状： (16918, 6)
标签值 (y) 预览：
600     6683426742
957     6683426742
4345    6889790653
4735    6822359752
5580    1527921905
Name: place_id, dtype: int64
标签形状： (16918,)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 划分数据集，测试集大小设置为 0.2，随机种子设置为42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 进行标准化处理
scaler = StandardScaler()  # 创建标准化缩放器
X_train_scaled = scaler.fit_transform(X_train)  # 拟合训练集并进行标准化变换
X_test_scaled = scaler.transform(X_test)  # 用同样的缩放器对测试集进行标准化变换



In [5]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 创建KNN分类器，这里假设使用5个邻居（可调整）
knn = KNeighborsClassifier(n_neighbors=5)

# 训练模型
knn.fit(X_train_scaled, y_train)

# 使用测试集进行预测
y_pred = knn.predict(X_test_scaled)

# 输出预测结果
print("前10个预测结果:", y_pred[:10])  # 输出预测结果的前10个
print("前10个真实标签:", y_test.iloc[:10].values)  # 输出真实标签的前10个，先取前10条再转为ndarray对象

# 输出准确率
accuracy = accuracy_score(y_test, y_pred)
print("KNN分类器的准确率：", accuracy)


前10个预测结果: [4932578245 5606572086 3992589015 3333445626 4932578245 3312463746
 4423196276 3686273628 1097200869 6683426742]
前10个真实标签: [6399991653 3862706892 3992589015 1893548673 7803770431 3312463746
 9598377925 6164537747 1097200869 6683426742]
KNN分类器的准确率： 0.5227541371158393


## 网格搜索 + 交叉验证 找到最佳超参数

In [6]:
from sklearn.model_selection import GridSearchCV

# 定义参数网格
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]  # 曼哈顿距离和欧氏距离
}

# 创建KNN分类器对象
knn = KNeighborsClassifier()

# 设置GridSearchCV
grid_search = GridSearchCV(
    estimator=knn,  # 指定KNN分类器对象
    param_grid=param_grid,  # 指定参数网格
    cv=5,  # 指定交叉验证的折数
    scoring='accuracy',  # 指定评分标准为准确率
    n_jobs=-1  # 指定使用所有CPU核心进行并行计算
)

# 在训练集上进行网格搜索
grid_search.fit(X_train_scaled, y_train)

# 输出最佳参数和最佳分数
print("最佳超参数：", grid_search.best_params_)
print("最佳交叉验证准确率：", grid_search.best_score_)

# 用最佳模型对测试集进行预测并评估
best_knn = grid_search.best_estimator_
y_pred_best = best_knn.predict(X_test_scaled)
accuracy_best = accuracy_score(y_test, y_pred_best)
print("使用最佳超参数的测试集准确率：", accuracy_best)


c:\Users\MSI-NB\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


最佳超参数： {'n_neighbors': 9, 'p': 1, 'weights': 'distance'}
最佳交叉验证准确率： 0.551204195085911
使用最佳超参数的测试集准确率： 0.556146572104019
